# VocEd Lab 7 — The Multi-Layer Perceptron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab7_multilayer_perceptron.ipynb)

Chapter 6 built neural networks in the TensorFlow Playground, on toy datasets with two features
and two classes.  This lab trains one on our project: three features per pixel (R, G, B) and three
classes.  Lab 6 measured the **colour ceiling**, the best any rule that sees one pixel's colour can
do, and found the random forest already at it.  Can a neural network do better?

- *What is inside the network?*  We count its weights and run its **forward pass** by hand.
- *What does each choice do?*  The activation, the random start and the size of the network,
  each tried on validation images.
- *Where is the limit?*  The network against Lab 6's colour ceiling, then one final test.

**Textbook:** [Chapter 6](https://obooks.tech/imageprocessing/chapter6.html), especially
[§6.13 Back to Our Cells](https://obooks.tech/imageprocessing/chapter6.html#back-to-our-cells-an-mlp-on-pixel-colour).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

Some cells take a minute to run; they say so.

## 0. Setup — Get the Data

The same images and split as Labs 3–6, and the same **random** training table as Labs 5 and 6:
450 pixels from anywhere in each of the 120 training images.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–6"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

def pixels_of(i):
    return X[i].transpose(1, 2, 0).reshape(-1, 3)                           # (65536, 3)

def predict_mask(classifier, i):
    return classifier.predict(pixels_of(i)).reshape(256, 256)

def val_dice(classifier):
    return np.mean([image_dice(predict_mask(classifier, i), y[i]) for i in val_idx])

# Lab 5's random training table
rng = np.random.default_rng(0)
rows, row_labels = [], []
for i in train_idx:
    chosen = rng.choice(65536, 450, replace=False)
    rows.append(pixels_of(i)[chosen])
    row_labels.append(y[i].reshape(-1)[chosen])
X_px, y_px = np.vstack(rows), np.concatenate(row_labels)
print("Training table:", X_px.shape)

**Scores so far, on the 40 validation images:** Lab 4 thresholds + clean-up 0.812;
Lab 5 logistic regression about 0.80; Lab 6 random forest 0.818, at the colour ceiling.

## 1. The Network

Our network is a **multi-layer perceptron (MLP)**: 3 input nodes (R, G, B), a hidden layer of 32
nodes, a hidden layer of 16, and 3 output nodes, one per class: 3 → 32 → 16 → 3.  Each hidden node
takes a weighted sum of the layer before, adds a bias, and applies **ReLU**.  The three outputs
go through a **softmax**, which turns them into three probabilities.  *(About 15 seconds.)*

### 🧩 Puzzle 1 — Count the weights

Between two layers, every node is connected to every node of the next layer, and each node in a
hidden or output layer also has one bias.  Fill in the middle connection.

📖 Book: [§6.13.1 The network: 3 → 32 → 16 → 3](https://obooks.tech/imageprocessing/chapter6.html#the-network-3-32-16-3)

<details><summary><b>💡 Hint</b> (click to open)</summary>

3 inputs to 32 nodes is 3 × 32 weights.  What connects 32 nodes to 16?

</details>

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=200, random_state=0).fit(X_px, y_px)
print(f"MLP validation Dice: {val_dice(mlp):.4f}")

n_weights = 3 * 32 + ___ + 16 * 3
n_biases  = 32 + 16 + 3
print(f"Weights {n_weights} + biases {n_biases} = {n_weights + n_biases} numbers to learn")
print("Logistic regression in Lab 5 learned 3 × 3 weights + 3 constants = 12")

# ── Check ──
assert n_weights == sum(W.size for W in mlp.coefs_), "One weight for every pair of connected nodes"
print("✅ Puzzle 1 solved!")

## 2. The Forward Pass by Hand

`mlp.coefs_` holds the three weight matrices and `mlp.intercepts_` the three bias vectors.  With
them we can push pixels through the network ourselves, one layer at a time, and check that we get
the same probabilities as `mlp.predict_proba`.

Each row of `x` is one pixel.  `x @ W1` multiplies it by the first weight matrix: one weighted sum
for each of the 32 nodes.  ReLU is `np.maximum(0, …)`: negative values become 0.

### 🧩 Puzzle 2 — The second hidden layer

📖 Book: [§6.5 Anatomy of a Neural Network](https://obooks.tech/imageprocessing/chapter6.html#anatomy-of-a-neural-network)

<details><summary><b>💡 Hint</b> (click to open)</summary>

The second layer does to `h1` exactly what the first layer did to `x`, with `W2` and `b2`.

</details>

In [ ]:
W1, W2, W3 = mlp.coefs_
b1, b2, b3 = mlp.intercepts_
print("Weight matrices:", W1.shape, W2.shape, W3.shape)

x  = pixels_of(val_idx[0])[[0, 30000, 32896]]          # three pixels of one validation image
h1 = np.maximum(0, x @ W1 + b1)                        # (3, 32)
h2 = np.maximum(0, ___)                        # (3, 16)
scores = h2 @ W3 + b3                                  # (3, 3): one score per class
probs  = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)    # softmax

print("Colours (R, G, B):")
print(x.round(2))
print("Probabilities (background, cytoplasm, nucleus):")
print(probs.round(3))
print("Hidden nodes switched off by ReLU in layer 1:", (h1 == 0).sum(axis=1), "of 32")

# ── Check ──
assert np.allclose(probs, mlp.predict_proba(x)), "Same weights, same steps, same probabilities"
print("✅ Puzzle 2 solved!")

## 3. Does the Activation Matter?

Chapter 6 found two things in the Playground: without a non-linear activation, hidden layers add
no power; and ReLU trains faster than sigmoid.  Do both hold on our pixels?  We train the same
3 → 32 → 16 → 3 network three times, changing only the activation:

- `'identity'`: no activation at all — each node passes its weighted sum straight on;
- `'logistic'`: the sigmoid;
- `'relu'`: ReLU, as above.

`loss_curve_` holds the training loss after every epoch.  *(About 40 seconds.)*

📖 Book: [§6.10.1 Question 3: Comparing ReLU, Sigmoid, and Tanh](https://obooks.tech/imageprocessing/chapter6.html#question-3-comparing-relu-sigmoid-and-tanh)

In [ ]:
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')        # the sigmoid network stops at 200 epochs before it has converged

nets = {}
for act in ['identity', 'logistic', 'relu']:
    nets[act] = MLPClassifier(hidden_layer_sizes=(32, 16), activation=act, max_iter=200, random_state=0).fit(X_px, y_px)
    print(f"{act:9s}  epochs {nets[act].n_iter_:3d}   final loss {nets[act].loss_:.4f}   validation Dice {val_dice(nets[act]):.4f}")

lr = LogisticRegression(max_iter=1000).fit(X_px, y_px)
print(f"logistic regression (Lab 5)              validation Dice {val_dice(lr):.4f}")

for act in nets:
    plt.plot(nets[act].loss_curve_, label=act)
plt.xlabel('epoch')
plt.ylabel('training loss')
plt.legend()
plt.show()

### 🤔 Think 1 — Three activations

1. Compare the `'identity'` network with logistic regression.  It has 707 numbers to learn instead
   of 12.  Why does it score the same?
2. Which network's loss falls fastest?  Which one is still falling when training stops?
3. Chapter 6 asked the same questions in the Playground.  Do our pixels give the same answers?

📖 Book: [§6.9.3 Question 2B: Linear Activation Cannot Learn Nonlinearity](https://obooks.tech/imageprocessing/chapter6.html#question-2b-linear-activation-cannot-learn-nonlinearity)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. Random Starts

Before training, the weights are set to small **random** numbers.  `random_state` picks which
random numbers.  Training then follows the loss downhill from wherever it started.  Does the
starting point matter?  *(About 40 seconds.)*

### 🧩 Puzzle 3 — How far apart are five runs?

📖 Book: [§6.11.3 Question 4: Does Convergence Always Look the Same?](https://obooks.tech/imageprocessing/chapter6.html#question-4-does-convergence-always-look-the-same)

<details><summary><b>💡 Hint</b> (click to open)</summary>

The spread is the largest score minus the smallest.  `max` and `min` work on a list.

</details>

In [ ]:
scores = []
for seed in range(5):
    net = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=200, random_state=seed).fit(X_px, y_px)
    scores.append(val_dice(net))
    print(f"random_state = {seed}   epochs {net.n_iter_:3d}   validation Dice {scores[-1]:.4f}")

spread = ___
print(f"Spread of the five validation Dice scores: {spread:.4f}")

# ── Check ──
assert np.isclose(spread, np.ptp(scores)), "Largest score minus smallest score"
print("✅ Puzzle 3 solved!")

### 🤔 Think 2 — Which run is the real one?

1. The five runs differ only in their random start.  How big is the spread, compared with the gap
   between the MLP and logistic regression?
2. Suppose you tried five settings of a knob, one run each, and the best beat the rest by 0.003.
   Would you trust that winner?  What would you do instead?

📖 Book: [§6.11 The Role of Initialization and Randomness](https://obooks.tech/imageprocessing/chapter6.html#the-role-of-initialization-and-randomness)

✏️ **Your answer:** *(double-click this cell and type here)*

## 5. Bigger Networks and the Colour Ceiling

Lab 6 measured the colour ceiling: on the validation images, the best rule that gives every colour
its most common label reaches a **pixel accuracy** of about 0.898.  No colour-only classifier can
beat it.  The cell below recomputes it (the same code as Lab 6), then trains four networks of
different sizes and compares each with the ceiling.  *(About a minute.)*

### 🧩 Puzzle 4 — How many numbers does each network learn?

`net.coefs_` is the list of weight matrices and `net.intercepts_` the list of bias vectors.  Count
the numbers in both.

📖 Book: [§6.12.3 Question 6: Finding the Minimal Architecture for Circles](https://obooks.tech/imageprocessing/chapter6.html#question-6-finding-the-minimal-architecture-for-circles)

<details><summary><b>💡 Hint</b> (click to open)</summary>

Puzzle 1 counted the weights with `sum(W.size for W in mlp.coefs_)`.  Do the same for the biases.

</details>

In [ ]:
# The colour ceiling, as in Lab 6
def colour_bin(pixels):
    levels = (np.minimum(pixels, 0.9999) * 32).astype(int)
    return levels[:, 0] * 32 * 32 + levels[:, 1] * 32 + levels[:, 2]

bins   = np.concatenate([colour_bin(pixels_of(i)) for i in val_idx])
labels = np.concatenate([y[i].reshape(-1) for i in val_idx])
counts = np.zeros((32768, 3), dtype=int)
np.add.at(counts, (bins, labels), 1)
ceiling_acc = np.mean(counts.argmax(axis=1)[bins] == labels)
print(f"Colour ceiling: pixel accuracy {ceiling_acc:.4f}")
print()

for size in [(2,), (8,), (32, 16), (128, 64)]:
    net = MLPClassifier(hidden_layer_sizes=size, max_iter=200, random_state=0).fit(X_px, y_px)
    n_numbers = sum(W.size for W in net.coefs_) + ___
    acc = np.mean(np.concatenate([net.predict(pixels_of(i)) for i in val_idx]) == labels)
    print(f"{str(size):10s} {n_numbers:6,d} numbers   pixel accuracy {acc:.4f}   validation Dice {val_dice(net):.4f}")

# ── Check ──
assert n_numbers == 3 * 128 + 128 * 64 + 64 * 3 + 128 + 64 + 3, "Weights plus biases"
print("✅ Puzzle 4 solved!")

### 🤔 Think 3 — More numbers, same wall

1. Going from 17 to 707 learned numbers, how much does validation Dice change?  From 707 to almost
   9,000?
2. Compare the networks' pixel accuracy with the ceiling.  How much room is left for a bigger
   network, however large?
3. Chapter 6's circle dataset *needed* a hidden layer: no straight line separates the classes.
   What is different about our pixels?

📖 Book: [§6.13.4 Why the network meets the same ceiling](https://obooks.tech/imageprocessing/chapter6.html#why-the-network-meets-the-same-ceiling)

✏️ **Your answer:** *(double-click this cell and type here)*

## 6. The Final Test — Once

The 3 → 32 → 16 → 3 network from Section 1, on the 40 test images.

📖 Book: [§6.13.3 Results](https://obooks.tech/imageprocessing/chapter6.html#results)

In [ ]:
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

preds   = [predict_mask(mlp, i) for i in test_idx]
dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
nc_pred = np.array([nc_ratio(p) for p in preds])
r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
close   = np.mean(np.abs(nc_pred - nc_true) < 0.05)

print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
print(f"{'MLP (3 → 32 → 16 → 3)':28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")
print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 4 specks, fill              0.772     -0.465   35%")
print("  Lab 5 k-NN, k = 25              0.763     -0.341   30%")
print("  Lab 6 random forest (leaf 100)  0.768     -0.412   32%")

### 🤔 Think 4 — A neural network on one pixel

1. How does the MLP compare with Lab 4, Lab 5's k-NN and Lab 6's forest on the test images?
2. Chapter 6 calls the network the more powerful tool.  Why did that power not show here?
3. What would you change about the network's *input* to give it something new to learn from?

📖 Book: [§5.9 Why All Colour-Only Methods Hit a Ceiling](https://obooks.tech/imageprocessing/chapter5.html#why-all-colour-only-methods-hit-a-ceiling)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **An MLP is layers of weighted sums with an activation in between.**  The forward pass is a few
  matrix multiplications, a ReLU after each hidden layer, and a softmax at the end.
- **The activation is what makes the layers count.**  Without one, 707 numbers describe the same
  flat planes as logistic regression's 12.
- **ReLU trains faster than sigmoid**, as in the Playground.
- **Random starts move the score.**  Compare settings over several seeds before declaring a winner.
- **Capacity is not information.**  Past a few hundred numbers, a bigger network does not help: it is
  already at the colour ceiling.  Lab 8 changes the input instead.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run |
|---|---|
| A | Section 3 with `'tanh'` added to the list of activations |
| B | Section 4 with `hidden_layer_sizes=(8,)` instead of `(32, 16)` |
| C | Section 5 with `early_stopping=True` added to the `MLPClassifier` |

1. Where does tanh fall between sigmoid and ReLU, in speed and in Dice?
2. Is a small network more or less sensitive to its random start than a big one?
3. `early_stopping=True` holds back 10% of the training table and stops when the score on it stops
   improving.  How does that change the number of epochs, and the Dice?